Hour 1 — Python Modules

Open src in VS Code and create a new file:

try importing the function with:

In [2]:
import os

print(os.getcwd())

c:\Users\gilbe\OneDrive\Desktop\30 Day DS DE\notebooks


In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [10]:
from src.pipeline import extract_data

Use your imported extract_data() function to load:

In [5]:
sales_df = extract_data("../data/day_06_pipeline_output.csv")

In [6]:
sales_df.head()

,order_id,product,quantity,price,order_date,total_sales
0,1001,Laptop,1.0,1200,2026-01-05,1200.0
1,1002,Monitor,2.0,300,2026-01-07,600.0
2,1003,Keyboard,1.0,100,2026-01-10,100.0
3,1004,Laptop,1.0,1200,2026-02-02,1200.0
4,1005,Monitor,3.0,300,2026-02-15,900.0


In a new notebook cell, try:

In [7]:
clean_sales_data(sales_df)

NameError: name 'clean_sales_data' is not defined

In [11]:
from src.pipeline import extract_data, clean_sales_data

ImportError: cannot import name 'clean_sales_data' from 'src.pipeline' (c:\Users\gilbe\OneDrive\Desktop\30 Day DS DE\src\pipeline.py)

In [15]:
import importlib
import src.pipeline

importlib.reload(src.pipeline)

<module 'src.pipeline' from 'c:\\Users\\gilbe\\OneDrive\\Desktop\\30 Day DS DE\\src\\pipeline.py'>

In [ ]:
from src.pipeline import extract_data, clean_sales_data

In [14]:
clean_df = clean_sales_data(sales_df)
clean_df.head()

,order_id,product,quantity,price,order_date,total_sales
0,1001,Laptop,1.0,1200,2026-01-05,1200.0
1,1002,Monitor,2.0,300,2026-01-07,600.0
2,1003,Keyboard,1.0,100,2026-01-10,100.0
3,1004,Laptop,1.0,1200,2026-02-02,1200.0
4,1005,Monitor,3.0,300,2026-02-15,900.0


Hour 2 — Move More ETL Logic Into pipeline.py

In [16]:
importlib.reload(src.pipeline)

<module 'src.pipeline' from 'c:\\Users\\gilbe\\OneDrive\\Desktop\\30 Day DS DE\\src\\pipeline.py'>

In [17]:
from src.pipeline import extract_data, clean_sales_data, validate_data

In [18]:
validate_data(sales_df)

{'duplicate_orders': np.int64(0),
 'invalid_values': np.int64(0),
 'missing_values': np.int64(0)}

Now add your load function to src/pipeline.py:

In [19]:
importlib.reload(src.pipeline)

<module 'src.pipeline' from 'c:\\Users\\gilbe\\OneDrive\\Desktop\\30 Day DS DE\\src\\pipeline.py'>

In [20]:
from src.pipeline import extract_data, clean_sales_data, validate_data, load_data

Now we can assemble the full reusable pipeline inside src/pipeline.py.

In [23]:
importlib.reload(src.pipeline)

<module 'src.pipeline' from 'c:\\Users\\gilbe\\OneDrive\\Desktop\\30 Day DS DE\\src\\pipeline.py'>

In [24]:
from src.pipeline import run_pipeline

In [25]:
day_09_df = run_pipeline("../data/day_06_pipeline_output.csv","../data/day_09_pipeline_output.csv")

day_09_df contains the expected data.  
The output CSV was actually created.

In [29]:
Path("../data/day_09_pipeline_output.csv").exists()

True

In [30]:
day_09_df.head()

,order_id,product,quantity,price,order_date,total_sales
0,1001,Laptop,1.0,1200,2026-01-05,1200.0
1,1002,Monitor,2.0,300,2026-01-07,600.0
2,1003,Keyboard,1.0,100,2026-01-10,100.0
3,1004,Laptop,1.0,1200,2026-02-02,1200.0
4,1005,Monitor,3.0,300,2026-02-15,900.0


Hour 3 — Better Path Management

In [31]:
data_dir = project_root / "data"

Create two variables:  
input_path  
output_path  

In [32]:
input_path = data_dir / "day_06_pipeline_output.csv"

In [33]:
output_path = data_dir / "day_09_pipeline_output_v2.csv"

Now run your reusable pipeline again, but this time use the Path variables instead of writing any file paths manually.  

Store the result in:

In [34]:
pipeline_result = run_pipeline(input_path,output_path)

Let's verify the new pipeline run.  
 
Check that:  

output_path exists.  
pipeline_result has the expected number of rows and columns.  

In [37]:
output_path.exists()

True

In [36]:
pipeline_result.shape

(5, 6)

Hour 4 — Make the Pipeline More Production-Like

In [40]:
importlib.reload(src.pipeline)

from src.pipeline import run_pipeline

In [41]:
pipeline_result = run_pipeline(input_path, output_path)

2026-09-01 23:48:14,286 - INFO - Starting pipeline
2026-09-01 23:48:14,292 - INFO - Extracting data
2026-09-01 23:48:14,293 - INFO - Validating data
2026-09-01 23:48:14,295 - INFO - Cleaning data
2026-09-01 23:48:14,297 - INFO - Loading data
2026-09-01 23:48:14,298 - INFO - Pipeline completed


In [42]:
import pandas as pd

bad_data = pd.DataFrame({
    "order_id": [1001, 1001],
    "product": ["Laptop", "Monitor"],
    "quantity": [1, 2],
    "price": [1200, 300],
    "order_date": ["2026-01-05", "2026-01-06"]
})

In [43]:
bad_data.to_csv("../data/day_09_bad_data.csv",index=False)

Now let's test whether your pipeline actually protects you from bad data.

In [44]:
run_pipeline(
    "../data/day_09_bad_data.csv",
    "../data/day_09_bad_output.csv"
)

2026-09-01 23:55:29,347 - INFO - Starting pipeline
2026-09-01 23:55:29,349 - INFO - Extracting data
2026-09-01 23:55:29,352 - INFO - Validating data


ValueError: Duplicate order IDs detected